# Clustering Data Final Clean

Notebook ridotto al solo flusso necessario delle sezioni originali `8.5)` e `8.6)` di `clustering_data.ipynb`.

Contenuto mantenuto:
- diagnostica dei ref-bin in spazio nucleotidi;
- export `all-pairs` nello stesso ref-bin con output reciprocal;
- export baseline completo `DT+ -> tutte le DT-`.

Contenuto rimosso:
- tutte le sezioni precedenti non necessarie a questo flusso;
- tutta la logica in spazio binario;
- metodi Top-N, reciprocal Top-N, KMeans, PCA e materiale legacy.

Il comportamento operativo resta quello del notebook originale per il percorso in spazio nucleotidi. La sola differenza e organizzativa: il codice e stato ripulito, accorpato e documentato.


## Sezione 1 - Import e utility minime

Qui definiamo solo cio che serve al flusso finale:
- import dei FASTA;
- scrittura dei CSV nel formato usato dal notebook originale;
- assegnazione delle sequenze ai ref-bin.

Nota importante: la diagnostica usa i bin standard `1-8, 9-12, ..., 57-60`, mentre l'export all-pairs aggiunge automaticamente un bin `0-0` solo se compaiono distanze nulle. Questo replica esattamente il comportamento originale.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pathlib import Path
from IPython.display import display

from adabmDCA.fasta import get_tokens, import_from_fasta


def write_index_rows_csv(path: Path, rows, expected_rows=None, verify_on_disk: bool = True) -> None:
    rows_list = [np.asarray(row, dtype=int).ravel() for row in rows]

    if expected_rows is not None and len(rows_list) != int(expected_rows):
        raise ValueError(
            f"Numero righe in memoria non valido per {path}: "
            f"atteso={int(expected_rows)}, trovato={len(rows_list)}"
        )

    with path.open("w", encoding="utf-8") as handle:
        for values in rows_list:
            handle.write(",".join(map(str, values.tolist())) + "\n")

    if verify_on_disk and expected_rows is not None:
        with path.open("r", encoding="utf-8") as handle:
            n_lines = sum(1 for _ in handle)
        if n_lines != int(expected_rows):
            raise ValueError(
                f"Numero righe su disco non valido per {path}: "
                f"atteso={int(expected_rows)}, trovato={n_lines}"
            )


def assign_distance_bins(distances: np.ndarray, bins, *, label: str, add_zero_bin_if_needed: bool = False):
    distances = np.asarray(distances, dtype=np.int32).ravel()
    if np.any(distances < 0):
        raise ValueError(f"[{label}] Trovate distanze negative")

    bins_used = list(bins)
    if add_zero_bin_if_needed and np.any(distances == 0):
        bins_used = [(0, 0)] + bins_used

    bin_ids = np.full(distances.shape[0], fill_value=-1, dtype=np.int32)
    idx_by_bin = []
    bin_labels = []

    for bin_idx, (lower, upper) in enumerate(bins_used):
        mask = (distances >= lower) & (distances <= upper)
        indices = np.where(mask)[0].astype(np.int32)
        idx_by_bin.append(indices)
        bin_labels.append(f"{lower}-{upper}")
        bin_ids[indices] = bin_idx

    if np.any(bin_ids < 0):
        uncovered = np.unique(distances[bin_ids < 0])
        raise ValueError(
            f"[{label}] Alcune distanze non rientrano nei bin definiti. "
            f"Valori non coperti: {uncovered.tolist()}"
        )

    return bin_ids, idx_by_bin, bin_labels, bins_used


## Sezione 2 - Configurazione e caricamento dataset

Questa sezione imposta i percorsi, carica `DN`, `DT-` e `DT+`, controlla che siano compatibili e prepara la reference `DN[0]` usata per tutte le distanze in spazio nucleotidi.


In [ ]:
DN_FASTA_PATH = Path("../data/RF00028_aligned_no_inserts_reweighted_d10.fa")
DTM_FASTA_PATH = Path(
    "../data/split_train_validation/"
    "split_0.15_vae25-30-in-val_nll-filtered_100/"
    "DTm_val_plus-vae25-30.fasta"
)
DTP_FASTA_PATH = Path(
    "../data/split_train_validation/"
    "split_0.15_vae25-30-in-val_nll-filtered_100/"
    "DTp_val_plus-vae25-30.fasta"
)

OUTPUT_CLUSTER_DIR = Path(
    "../data/pairings/"
    "split_0.15_vae25-30-in-val_nll-filtered_100/"
    "pair_validation/"
)
OUTPUT_CLUSTER_DIR.mkdir(parents=True, exist_ok=True)

FILE_ALLPAIRS_NT_REFBIN_RECIPROCAL = "allpairs_nt_refbin_reciprocal.csv"
FILE_ALLPAIRS_REFBIN_STATS_TXT = "allpairs_refbin_stats.txt"
FILE_ALL_DTM_FOR_EACH_DTP = "all_pairs.csv"

print("Config pronta:")
print(" DN  :", DN_FASTA_PATH)
print(" DT- :", DTM_FASTA_PATH)
print(" DT+ :", DTP_FASTA_PATH)
print(" OUT :", OUTPUT_CLUSTER_DIR)


tokens = get_tokens("rna")

headers_DN, data_DN = import_from_fasta(str(DN_FASTA_PATH), tokens=tokens, filter_sequences=True)
headers_DTm, data_DTm = import_from_fasta(str(DTM_FASTA_PATH), tokens=tokens, filter_sequences=True)
headers_DTp, data_DTp = import_from_fasta(str(DTP_FASTA_PATH), tokens=tokens, filter_sequences=True)

print("Shape DN :", data_DN.shape)
print("Shape DT-:", data_DTm.shape)
print("Shape DT+:", data_DTp.shape)

if data_DN.shape[0] == 0:
    raise ValueError("DN e vuoto: impossibile usare la prima sequenza come reference")

if not (data_DN.shape[1] == data_DTm.shape[1] == data_DTp.shape[1]):
    raise ValueError("DN, DT- e DT+ devono avere la stessa lunghezza allineata")

DTm = data_DTm
DTp = data_DTp
n_tm = DTm.shape[0]
n_tp = DTp.shape[0]

reference_seq = data_DN[0]


## Sezione 3 - Diagnostica dei ref-bin

Questa e la parte diagnostica collegata al flusso finale. Calcoliamo la distanza dalla reference per `DT-` e `DT+` in spazio nucleotidi e contiamo quante sequenze cadono in ogni bin.

I bin usati qui sono quelli standard del notebook originale:
- `1-8`;
- `9-12`, `13-16`, ..., `57-60`.


In [ ]:
distance_bins = [(1, 8)] + [(start, min(start + 3, 60)) for start in range(9, 61, 4)]
bin_labels = [f"{lower}-{upper}" for lower, upper in distance_bins]
x = np.arange(len(distance_bins))
width = 0.42


dist_tm_ref_nt = (DTm != reference_seq).sum(axis=1).astype(np.int32)
dist_tp_ref_nt = (DTp != reference_seq).sum(axis=1).astype(np.int32)

tm_bins_nt, _, _, _ = assign_distance_bins(
    dist_tm_ref_nt,
    distance_bins,
    label="diagnostic|nucleotide|DT-",
)
tp_bins_nt, _, _, _ = assign_distance_bins(
    dist_tp_ref_nt,
    distance_bins,
    label="diagnostic|nucleotide|DT+",
)

counts_tm_nt = np.bincount(tm_bins_nt, minlength=len(distance_bins))
counts_tp_nt = np.bincount(tp_bins_nt, minlength=len(distance_bins))

plt.figure(figsize=(15, 5))
plt.bar(x - width / 2, counts_tm_nt, width=width, color="tab:red", alpha=0.85, label="DT-")
plt.bar(x + width / 2, counts_tp_nt, width=width, color="tab:green", alpha=0.85, label="DT+")
plt.title("Distribuzione ref-bin (nucleotide space)")
plt.ylabel("Count")
plt.xlabel("Ref-bin")
plt.xticks(x, bin_labels, rotation=45, ha="right")
plt.grid(True, axis="y", alpha=0.25)
plt.legend(loc="best")
plt.show()

print("Conteggi per bin (nucleotide):")
for label, count_tm, count_tp in zip(bin_labels, counts_tm_nt, counts_tp_nt):
    print(f"  bin {label}: DT-={int(count_tm)}, DT+={int(count_tp)}")


df_refbin_counts_current = pd.DataFrame(
    {
        "ref_bin": bin_labels,
        "positive_DT+_nt": counts_tp_nt.astype(int),
        "negative_DT-_nt": counts_tm_nt.astype(int),
    }
)

print("\nConteggi DT+/DT- per ref-bin (binning corrente):")
display(df_refbin_counts_current)


## Sezione 4 - All-pairs nello stesso ref-bin con export reciprocal

Questa e la parte principale della sezione originale `8.5)`.

Il flusso e il seguente:
1. assegniamo `DT+` e `DT-` ai ref-bin in spazio nucleotidi;
2. per ogni `DT+` prendiamo tutte le `DT-` dello stesso bin;
3. aggiungiamo il passaggio reciproco, cioe tutte le `DT-` che vedono quella `DT+` nel proprio stesso bin;
4. salviamo un CSV con una riga per ogni `DT+` e tutti gli indici `DT-` associati.

Qui il bin `0-0` viene aggiunto automaticamente solo se necessario, esattamente come nel notebook originale.


In [ ]:
allpairs_distance_bins = [(1, 8)] + [(start, min(start + 3, 60)) for start in range(9, 61, 4)]


def build_allpairs_reciprocal_same_bin(tp_bin_ids: np.ndarray, tm_bin_ids: np.ndarray, tp_idx_by_bin, tm_idx_by_bin):
    n_tp_local = tp_bin_ids.shape[0]
    n_bins = len(tp_idx_by_bin)

    rows_direct = []
    direct_counts_by_bin = np.zeros(n_bins, dtype=np.int64)
    for tp_idx in range(n_tp_local):
        bin_idx = int(tp_bin_ids[tp_idx])
        tm_candidates = tm_idx_by_bin[bin_idx]
        rows_direct.append(tm_candidates.astype(np.int32, copy=False))
        direct_counts_by_bin[bin_idx] += int(tm_candidates.size)

    extras_per_tp = [set() for _ in range(n_tp_local)]
    reverse_counts_by_bin = np.zeros(n_bins, dtype=np.int64)
    for tm_idx in range(tm_bin_ids.shape[0]):
        bin_idx = int(tm_bin_ids[tm_idx])
        tp_candidates = tp_idx_by_bin[bin_idx]
        reverse_counts_by_bin[bin_idx] += int(tp_candidates.size)
        for tp_idx in tp_candidates:
            extras_per_tp[int(tp_idx)].add(int(tm_idx))

    rows_reciprocal = []
    added_counts_per_tp = []
    for tp_idx in range(n_tp_local):
        direct = [int(value) for value in rows_direct[tp_idx].tolist()]
        extra_sorted = sorted(extras_per_tp[tp_idx] - set(direct))
        merged = np.asarray(direct + extra_sorted, dtype=np.int32)
        rows_reciprocal.append(merged)
        added_counts_per_tp.append(len(extra_sorted))

    return (
        rows_reciprocal,
        direct_counts_by_bin,
        reverse_counts_by_bin,
        np.asarray(added_counts_per_tp, dtype=np.int32),
    )


(
    tp_bins_nt_ap,
    tp_idx_by_bin_nt_ap,
    bin_labels_nt_ap,
    bins_used_nt_ap,
) = assign_distance_bins(
    dist_tp_ref_nt,
    allpairs_distance_bins,
    label="allpairs|nucleotide|DT+",
    add_zero_bin_if_needed=True,
)
(
    tm_bins_nt_ap,
    tm_idx_by_bin_nt_ap,
    _,
    _,
) = assign_distance_bins(
    dist_tm_ref_nt,
    allpairs_distance_bins,
    label="allpairs|nucleotide|DT-",
    add_zero_bin_if_needed=True,
)

(
    rows_nt_allpairs_rec,
    direct_nt_by_bin,
    reverse_nt_by_bin,
    added_nt_per_tp,
) = build_allpairs_reciprocal_same_bin(
    tp_bin_ids=tp_bins_nt_ap,
    tm_bin_ids=tm_bins_nt_ap,
    tp_idx_by_bin=tp_idx_by_bin_nt_ap,
    tm_idx_by_bin=tm_idx_by_bin_nt_ap,
)

count_tp_nt_by_bin = np.asarray([len(indices) for indices in tp_idx_by_bin_nt_ap], dtype=np.int64)
count_tm_nt_by_bin = np.asarray([len(indices) for indices in tm_idx_by_bin_nt_ap], dtype=np.int64)
count_sum_nt_by_bin = count_tp_nt_by_bin + count_tm_nt_by_bin

path_allpairs_nt_rec = OUTPUT_CLUSTER_DIR / FILE_ALLPAIRS_NT_REFBIN_RECIPROCAL
path_allpairs_stats = OUTPUT_CLUSTER_DIR / FILE_ALLPAIRS_REFBIN_STATS_TXT

write_index_rows_csv(path_allpairs_nt_rec, rows_nt_allpairs_rec, expected_rows=n_tp)

with path_allpairs_stats.open("w", encoding="utf-8") as handle:
    handle.write("All-pairs same-bin report (reciprocal export)\n")
    handle.write("Bins target: 1-8, 9-12, ..., 57-60 (piu eventuale 0-0 se presente)\n\n")
    handle.write("[Nucleotide space]\n")
    handle.write("bin_label,n_seq_dt_plus,n_seq_dt_minus,sum_dt_plus_minus\n")
    for idx, label in enumerate(bin_labels_nt_ap):
        handle.write(
            f"{label},{int(count_tp_nt_by_bin[idx])},{int(count_tm_nt_by_bin[idx])},{int(count_sum_nt_by_bin[idx])}\n"
        )

print("All-pairs same-bin (reciprocal) completato.")
print("CSV nucleotide:", path_allpairs_nt_rec)
print("TXT stats     :", path_allpairs_stats)
print(f"Extra medi aggiunti da reciprocita (NT): {added_nt_per_tp.mean():.2f}")

allpairs_bins_used_nucleotide = bins_used_nt_ap
allpairs_bin_labels_nucleotide = bin_labels_nt_ap
allpairs_csv_nucleotide_reciprocal = path_allpairs_nt_rec
allpairs_stats_txt_path = path_allpairs_stats
allpairs_direct_counts_nucleotide = direct_nt_by_bin
allpairs_reverse_counts_nucleotide = reverse_nt_by_bin
allpairs_seqcount_dtplus_nucleotide = count_tp_nt_by_bin
allpairs_seqcount_dtminus_nucleotide = count_tm_nt_by_bin
allpairs_seqcount_sum_nucleotide = count_sum_nt_by_bin


## Sezione 5 - Baseline completa `DT+ -> tutte le DT-`

Questa e la sezione originale `8.6)`. Il comportamento e il piu semplice possibile: per ogni sequenza `DT+` scriviamo una riga con tutti gli indici di `DT-`.


In [ ]:
path_all_dtminus_for_each_dtplus = OUTPUT_CLUSTER_DIR / FILE_ALL_DTM_FOR_EACH_DTP
all_dtminus_indices = np.arange(n_tm, dtype=np.int32)
rows_all_dtminus_for_each_dtplus = [all_dtminus_indices.copy() for _ in range(n_tp)]

write_index_rows_csv(
    path_all_dtminus_for_each_dtplus,
    rows_all_dtminus_for_each_dtplus,
    expected_rows=n_tp,
)

print("CSV completo DT+ -> tutte le DT- salvato:")
print(path_all_dtminus_for_each_dtplus)
print(f"Numero righe DT+: {n_tp}")
print(f"Numero indici DT- per riga: {n_tm}")

all_dtminus_for_each_dtplus_csv_path = path_all_dtminus_for_each_dtplus
all_dtminus_for_each_dtplus_rows = rows_all_dtminus_for_each_dtplus
